# Forecast Regret
How much does a weather forecast's error cost a transatlantic flight? For each **flight** we plan on a forecast source's winds, fly that **plan** through the **truth**, and compare it with the **hindsight plan**, the best candidate with the truth known.
**Regret** is the extra fuel, time and CO₂ that plan costs over the hindsight plan.

In [ ]:
import os, time
from pathlib import Path

from regret import offline
offline.install_if_requested()
_t0 = time.perf_counter()

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(_root)  # the kernel starts in notebooks/; config paths are relative to the repo root

import pandas as pd
import plotly.offline
from IPython.display import HTML, display

from regret import config
from regret.figures import showcase_map
from regret.figures.headline import FIG_ID as HEADLINE_ID, build_headline
from regret.figures.showcase_map import FIG_ID as MAP_ID, build_showcase_map
from regret.figures.lead_regret import FIG_ID as LEAD_ID, build_lead_regret
from regret.figures.wind_error import FIG_ID as WIND_ID, build_wind_error
from regret.figures.sanity import FIG_ID as SANITY_ID, build_sanity

cfg = config.get_config()
df = pd.read_parquet(cfg.results_path)
plotly.offline.init_notebook_mode(connected=False)  # plotly.js loaded once, inline
display(HTML(showcase_map.geo_assets_html()))  # basemap preloaded, so nothing is fetched from a CDN


def show(fig, fig_id):
    display(HTML(fig.to_html(full_html=False, include_plotlyjs=False, div_id=fig_id,
                                 config={"displayModeBar": False})))

## 1. Headline: what forecast error costs
Mean (with 90% CI), median and p90 regret in fuel, time and CO₂, by forecast source and lead, eastbound and westbound flights pooled.
On a narrow screen, scroll the table sideways to see every column.

In [ ]:
show(build_headline(df), HEADLINE_ID)

## 2. Showcase flight: JFK→LHR, 2020-02-09
Every forecast source's plan over the truth's 250 hPa wind speed, with the hindsight plan drawn as a wide translucent band beneath them. Use the dropdown to pick the lead and the slider to see each source's forecast winds instead.

In [ ]:
show(build_showcase_map(cfg), MAP_ID)

## 3. Regret vs lead time
Mean fuel regret by lead; the top row shows the forecast sources (IFS HRES, GraphCast, Pangu-Weather), eastbound and westbound flights side by side. The persistence baseline sits on its own row below, on its own scale because its regret is far larger. Lines are means and shaded bands are 90% CIs.

In [ ]:
show(build_lead_regret(df), LEAD_ID)

## 4. Wind error vs fuel regret
The RMS wind error along the hindsight plan's path against the plan's fuel regret, in two panels, forecast sources and the persistence baseline, each on its own scale. Faint points are single plans and lines are each source's binned median; plans above the plotted range are pinned at the top edge, with their true value in the hover.

In [ ]:
show(build_wind_error(df), WIND_ID)

## 5. Sanity panel
Checks that the experiment behaves physically: eastbound vs westbound replay times, the fuel-flow distribution, and how often an optimiser gap occurs and how large it is.

In [ ]:
show(build_sanity(df), SANITY_ID)

## 6. Method, assumptions and limitations

**Method.**
- Winds vary in space and time along the flight: bilinear in latitude/longitude, linear in time between 6-hourly steps. A forecast plan uses one initialisation and steps through its prediction steps as the flight progresses.
- The planner is a dynamic programme on a lateral grid aligned with the great circle (25 columns × 41 cross-track offsets spaced 60 km, at most ±3 offsets between columns). It keeps the earliest arrival at each node, at constant TAS (M0.85, FL350, ISA). With TAS and flight level fixed, the minimum-time plan is also the minimum-fuel and minimum-CO₂ plan.
- Replay flies a plan's fixed lateral path through the truth in steps of about 10 km, with OpenAP B789 fuel flow at the current mass; CO₂ = 3.16 × fuel.
- For each flight the candidates are the truth-optimised plan and every forecast plan. The lowest replayed cost is the hindsight plan, and a plan's **regret** is its replayed cost minus the hindsight plan's cost, for fuel, time and CO₂ separately.
- Wind error is the RMS of |V_forecast − V_truth| at 250 hPa along the hindsight plan's path.

### Assumptions and limitations
- Cruise only at fixed FL350 and M0.85; no climb, descent or step climbs.
- No ATC, North Atlantic Tracks, airspace or ETOPS constraints.
- OpenAP fuel models are accurate to roughly ±5–10%; regret is a difference, so most bias cancels.
- Winds at 1.5° under-resolve the jet core; ERA5 is treated as truth but is itself a model.
- Small sample: 2 months, 1 city pair, 1 aircraft. Results show spread, not significance; CIs are given.
- ISA temperature (no temperature effect on TAS); no turbulence.
- The hindsight plan is the best of the candidates, so the planner's grid limits what "best" means.
- The IFS ensemble is unused (no 250 hPa in WeatherBench 2), and so is GenCast; GFS is not in WeatherBench 2.

In [ ]:
print(f"elapsed_s={time.perf_counter() - _t0:.1f}")